# Document Scanner Model Training

In this project, we train the models need for the document scanner and we create preprocessing tools for working with the images.

### Models Interacted with
- YOLO11 - Training the yolo 11 model to segement lines of handwritings
- TROCR-Handwritten - Working with TROCR Handwritten pretrained model for detecting words from images of handwritten documents (line segmented)

### Preprocessing Steps Carried out
#### Scan simulation
 1. Load with EXIF rotation correction
 2. Perspective correction (un-skew the document)
 3. Convert to grayscale
 4. Normalize uneven lighting (e.g. shadows from phone flash)
 5. Adaptive binarization (convert to pure black and white)
 6. Morphological cleanup (remove small noise specks)


### Mount Drive for storage during runtime

In [ ]:
##  mount Google Drive so we can read/write files
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


### Dependencies

In [ ]:

## Install dependencies
%pip install piexif     # for camera metadata(orientation info)
%pip install Pillow-heif      # For opening iPhone format images
%pip install ultralytics
%pip install roboflow
%pip install language-tool-python
%pip install pyspellchecker
%pip install ollama

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 85.6 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.13
    Uninstalling idna-3.13:
      Successfully uninstalled idna-3.13
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 kB 3.6 MB/s eta 0:00:00
   ━━━━

### Imports

In [ ]:
# TrOCRProcessor  : prepares images into the tensor format TrOCR expects
# VisionEncoderDecoderModel : the actual OCR model (vision encoder + text decoder)
from google.colab import userdata # For accessing secrets
from PIL import Image as PILImage   # Pillow — Python's standard image library
from pillow_heif import register_heif_opener      # Adds HEIC support to Pillow
from roboflow import Roboflow
from transformers import TrOCRProcessor , VisionEncoderDecoderModel
from ultralytics import YOLO   # Object detection
from spellchecker import SpellChecker


import language_tool_python
import openai
import numpy as np
import torch       #  used to run the model
import piexif      # Reads metadata embedded in JPEG files
import numpy as np
import cv2       # OpenCV — computer vision library for image processing
import matplotlib.pyplot as plt       # For visualising images
import os
import yaml
import zipfile
import ollama
from ollama import Client

# Allow Pillow to open HEIC files (common on iPhones)
register_heif_opener()

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [6]:
# Define Variable constants
BASE_DIR = "/content"
YOLO_DATA_DIR = os.path.join(BASE_DIR, 'Line-Segmentation-6')
YOLO_TRAIN_DATA_DIR = os.path.join(YOLO_DATA_DIR, 'train')
YOLO_VALID_DATA_DIR = os.path.join(YOLO_DATA_DIR, 'valid')
YOLO_TEST_DATA_DIR = os.path.join(YOLO_DATA_DIR, 'test')

# Handle secrets
#ROBOFLOW_API_KEY = userdata.get('api_key')
#WORKSPACE_NAME = userdata.get('workspace_name')
OLLAMA_API_KEY = userdata.get('ollama_api_key')

# Transformer model
# TrOCR is a Microsoft model trained specifically on handwritten text.
# "large-handwritten" is the biggest/most accurate variant.
TROCR_HANDWRITTEN_MODEL_NAME = 'microsoft/trocr-large-handwritten'

DEEPSEEK_KEY =  userdata.get('deepseek_secret_key')


### Load the OCR model

In [ ]:
processor = TrOCRProcessor.from_pretrained(TROCR_HANDWRITTEN_MODEL_NAME)
model = VisionEncoderDecoderModel.from_pretrained(TROCR_HANDWRITTEN_MODEL_NAME)

# Use a GPU if one is available — the model runs much faster on CUDA (Nvidia GPU)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)



The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.


preprocessor_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

The image processor of type `ViTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/2.23G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.23G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/635 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie decoder.model.decoder.embed_tokens.weight to decoder.output_projection.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-large-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.bias   | MISSING | 
encoder.pooler.dense.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## Helper Functions

In [ ]:
def order_points(pts):
    """
    Given 4 corner points in any order, return them sorted as:
    [top-left, top-right, bottom-right, bottom-left].

    OpenCV needs to know which corner maps to which destination point.

    Args:
        pts (np.ndarray): Array of shape (4, 2) — four (x, y) coordinates.

    Returns:
        np.ndarray: The same 4 points reordered as [TL, TR, BR, BL].
    """
    rect = np.zeros((4, 2), dtype="float32")
    s = pts.sum(axis=1)
    diff = np.diff(pts, axis=1)
    rect[0] = pts[np.argmin(s)]      # top-left
    rect[2] = pts[np.argmax(s)]      # bottom-right
    rect[1] = pts[np.argmin(diff)]   # top-right
    rect[3] = pts[np.argmax(diff)]   # bottom-left
    return rect

def load_with_exif_rotation(path):
    """
    Load an image from disk and apply any rotation encoded in its EXIF (photo) metadata.

    Args:
        path (str): Path to the image file.

    Returns:
        np.ndarray: BGR image array (OpenCV's native format), correctly rotated.
    """
    pil_img = PILImage.open(path)
    # Try to read the orientation tag from EXIF data
    try:
        exif_data = piexif.load(pil_img.info.get("exif", b""))
        orientation = exif_data["0th"].get(piexif.ImageIFD.Orientation, 1)
    except:
        orientation = 1 # Default: no rotation needed
    # EXIF orientation codes and their corresponding rotation angles
    rotation_map = {3: 180, 6: -90, 8: 90}
    if orientation in rotation_map:
        pil_img = pil_img.rotate(rotation_map[orientation], expand=True)
    # Convert from Pillow (RGB) to OpenCV (BGR) format
    return cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)

# un-skew the document
def perspective_correct(image):
    """
    Detect the document in the image and apply a perspective transform to make
    it appear as if photographed as a flat rectangular scan.

    HOW IT WORKS:
        1. Convert to grayscale and detect edges (Canny edge detector)
        2. Find contours (closed shapes) in the edge map
        3. Look for the largest 4-sided contour — that's likely the document
        4. Compute a perspective transform from those 4 corners to a rectangle
        5. Warp the image using that transform

    Args:
        image (np.ndarray): BGR image (as loaded by OpenCV).

    Returns:
        np.ndarray: Perspective-corrected BGR image, or the original if no
                    document contour was found.
    """
    orig = image.copy()
    #Grayscale + blur to reduce noise before edge detection
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    # Canny edge detector: finds sharp intensity changes (i.e. document borders)
    # The two numbers (75, 200) are the lower and upper thresholds
    edged = cv2.Canny(blurred, 75, 200)
    #Find all contours, keep the 5 largest by area
    contours, _ = cv2.findContours(edged, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
    contours = sorted(contours, key=cv2.contourArea, reverse=True)[:5]

    # Among those contours, find the first one that's roughly quadrilateral
    doc_contour = None
    for c in contours:
        # approxPolyDP simplifies the contour to fewer points
        # 0.02 * perimeter is the tolerance for how much the shape can deviate
        peri = cv2.arcLength(c, True)
        approx = cv2.approxPolyDP(c, 0.02 * peri, True)
        if len(approx) == 4:  # A page has 4 corners
            doc_contour = approx
            break

    if doc_contour is None:
        print("No document contour found — skipping perspective correction")
        return image  # Return the image unchanged

    # Order the 4 corners consistently (TL, TR, BR, BL)
    pts = order_points(doc_contour.reshape(4, 2))
    tl, tr, br, bl = pts

    # compute output rectangle dimensions
    widthA = np.linalg.norm(br - bl)   # Bottom edge width
    widthB = np.linalg.norm(tr - tl)   # Top edge width
    heightA = np.linalg.norm(tr - br)  # Right edge height
    heightB = np.linalg.norm(tl - bl)  # Left edge height
    maxW = int(max(widthA, widthB))
    maxH = int(max(heightA, heightB))

    # Define where the 4 corners should map to in the output (a flat rectangle)
    dst = np.array([
        [0, 0],
        [maxW - 1, 0],
        [maxW - 1, maxH - 1],
        [0, maxH - 1]
    ], dtype="float32")

    # Compute and apply the perspective transform
    M = cv2.getPerspectiveTransform(pts, dst)
    warped = cv2.warpPerspective(orig, M, (maxW, maxH))
    return warped


def universal_nlp_layer(raw_text):
    # 1. Immediate Visual Cleanup
    # Removes the "—" and weird symbols TrOCR adds at line edges
    clean_text = raw_text.replace("—", "").replace("“11", "I'll").strip()

    # 2. Contextual Spellcheck (The Upgrade)
    # We use a 'TextBlob' or 'SymSpell' approach which is better for context
    from textblob import TextBlob

    # TextBlob looks at the surrounding words to decide if "far" should be "for"
    blob = TextBlob(clean_text)
    contextual_text = str(blob.correct())

    # 3. LanguageTool Final Polish
    # Now that words are likely correct, we fix the grammar
    tool = language_tool_python.LanguageTool('en-US', language_tool_download_version='5.2')
    final_output = tool.correct(contextual_text)
    return final_output

    import openai

def deepseek_nlp_layer(raw_text):
    client = openai.OpenAI(api_key=DEEPSEEK_KEY, base_url="https://api.deepseek.com")

    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=[
            {"role": "system", "content": "You are an expert OCR post-processor. "
                                          "Restructure and fix errors in this handwritten transcription. "
                                          "Fix visual misreads (e.g., 'far' to 'for', '11' to 'I'll'). "
                                          "Maintain the original tone and line breaks."},
            {"role": "user", "content": raw_text}
        ]
    )
    return response.choices[0].message.content


from ollama import Client

def ollama_nlp_layer(raw_text):
    # Instructions for the model to handle your specific handwriting errors
    system_prompt = (
        "You are an expert OCR post-processor. "
        "Restructure and fix errors in this handwritten transcription. "
        "Fix visual misreads like 'far' -> 'for', '11' -> 'I'll', and 'votes' -> 'routes'. "
        "Maintain the original tone and line breaks. Output ONLY the corrected text."
    )

    # We use llama3.2 because it is fast and fits well in Colab's free GPU
    # (Make sure you ran !ollama pull llama3.2 previously)


    # Load the Ollama CLient
    client = Client(
      host="https://ollama.com",
      headers={'Authorization': 'Bearer ' + OLLAMA_API_KEY}
    )

    # Define the
    messages = [
      {
          'role': 'You are an expert Paleographer and Linguistic',
          'content': (
            "You are an expert OCR post-processor. "
            "Restructure and fix errors in this handwritten transcription. "
            "Fix visual misreads like 'far' -> 'for', '11' -> 'I'll', and 'votes' -> 'routes'. "
            "Maintain the original tone and line breaks. Output ONLY the corrected text."
        )
      },
      {
        'role': 'user',
        'content': raw_text,
      },
    ]

    response = client.chat('gpt-oss:120b', messages=messages)

    return response['message']['content']

# --- How to use it in your script ---
# final_output = ollama_nlp_layer(raw_full_text)
# print(final_output)


### Using OpenCV to give a scanned effect on an image

In [8]:
def process_document(image_path: str) -> np.ndarray:
    """
    Load and pre-process a document photo to produce a clean, binarized image
    that looks like a high-quality scan.

    WHY PRE-PROCESS?
    Raw photos have uneven lighting, perspective distortion, and colour noise.
    OCR models work much better on clean, high-contrast black-and-white images.

    Steps applied:
        1. Load with EXIF rotation correction
        2. Perspective correction (un-skew the document)
        3. Convert to grayscale
        4. Normalize uneven lighting (e.g. shadows from phone flash)
        5. Adaptive binarization (convert to pure black and white)
        6. Morphological cleanup (remove small noise specks)

    Args:
        image_path (str): Path to the input image.

    Returns:
        np.ndarray: A cleaned, binarized (black & white) image ready for OCR.
    """
    # Fix rotation and perspective
    image = load_with_exif_rotation(image_path)
    image = perspective_correct(image)

    # the OCR model doesn't need colour information
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    #  Normalize lighting
    # GaussianBlur with a large kernel (51x51) creates a smooth "background brightness" map.
    # Dividing the image by this map removes lighting gradients (e.g. one side darker than other).
    blur = cv2.GaussianBlur(gray, (51, 51), 0)
    norm = cv2.divide(gray, blur, scale=255)

   # Step 5: Adaptive thresholding — converts grayscale to pure black & white.
    # Unlike simple thresholding, ADAPTIVE_THRESH_GAUSSIAN_C computes a local
    # threshold for each pixel based on its neighbourhood. This handles cases
    # where different parts of the image have different brightness levels.
    binary = cv2.adaptiveThreshold(
        norm, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        51,   # Block size: size of the neighbourhood area (must be odd)
        15    # C: a constant subtracted from the mean to fine-tune the threshold
    )

    # Step 6: Morphological opening — erodes then dilates to remove tiny noise specks.
    # A small 2x2 kernel means only very small artefacts are removed.
    kernel = np.ones((2, 2), np.uint8)
    clean = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)

    return clean

# run TrOCR on a processed image
def run_ocr(clean: np.ndarray) -> str:
    """
    Run handwriting recognition on a pre-processed document image.

    HOW IT WORKS:
        1. Convert the NumPy array back to a PIL image in RGB mode
           (TrOCR expects a colour PIL image, even though ours is grayscale)
        2. The processor tokenizes the image into pixel_values tensors
        3. The model generates a sequence of token IDs (like a language model)
        4. The processor decodes those token IDs back into a string

    Args:
        clean (np.ndarray): A binarized grayscale image (output of process_document).

    Returns:
        str: The extracted text as a plain string.
    """
     # TrOCR expects a 3-channel (RGB) PIL image — convert even though it's B&W
    pil_image = PILImage.fromarray(clean).convert('RGB')

    # Preprocess: resize, normalize, and convert to a PyTorch tensor
    pixel_values = processor(pil_image, return_tensors='pt').pixel_values.to(device)

    # Generate text token IDs using the model (greedy decoding by default)
    generated_ids = model.generate(pixel_values)

    # Decode token IDs back to a human-readable string
    return processor.batch_decode(generated_ids, skip_special_tokens=True)[0]


### WORKING WITH YOLO11 TO DETECT LINES

### Get the Dataset from our Roboflow workspace

In [ ]:
# Get the dataset from the remote roboflow server

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE_NAME).project("line-segmentation-uqcgi-ullbn")
version = project.version(6)
dataset = version.download("yolov11")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Line-Segmentation-6 in yolov11:: 100%|██████████| 6767/6767 [00:05<00:00, 1273.23it/s]


### Verify the Dataset

In [ ]:
# Handle Extraction ---
# Most Roboflow versions auto-extract to a folder named 'line-segmentation-6'
# But if you have a zip file, use this:

dataset_path = dataset.location

if dataset_path.endswith('.zip'):
    with zipfile.ZipFile(dataset_path, 'r') as zip_ref:
        zip_ref.extractall(YOLO_DATA_DIR)
    config_path = os.path.join(YOLO_DATA_DIR, "data.yaml")
else:
    # If already a folder, find the data.yaml inside it
    config_path = os.path.join(dataset_path, "data.yaml")


In [ ]:
# Verify the dataset file structure
if not dataset_path.endswith('.zip'):
  path_data = os.listdir(dataset_path)
else:
  path_data = zip_ref.namelist()

path_data

['README.roboflow.txt',
 'test',
 'train',
 'README.dataset.txt',
 'valid',
 'data.yaml']

### Manipulate the data.yaml file to ensure it works with our workspace

In [ ]:
# Run this to ensure YOLO doesn't see "ghost" classes.

# Path to your extracted roboflow data.yaml
yaml_path = os.path.join(YOLO_DATA_DIR, 'data.yaml')

with open(yaml_path, 'r') as f:
    config = yaml.safe_load(f)

# Ensure these exact values
config['nc'] = 1

# Use absolute paths to prevent "No images found" errors
config['train'] = os.path.join(YOLO_TRAIN_DATA_DIR, 'images')
config['val'] = os.path.join(YOLO_VALID_DATA_DIR, 'images')
config['test'] = os.path.join(YOLO_VALID_DATA_DIR, 'images')

with open(yaml_path, 'w') as f:
    yaml.dump(config, f)

In [ ]:
print("Train images count:", len(os.listdir(os.path.join(YOLO_TRAIN_DATA_DIR, 'images'))))
print("Valid images count:", len(os.listdir(os.path.join(YOLO_VALID_DATA_DIR, 'images'))))

Train images count: 2958
Valid images count: 211


### Training the YOLO11 Model for Line Detection

In [ ]:
# --- Train YOLO11 ---
# Load the nano model (best for detection)
model = YOLO("yolo11n.pt")

model.train(
    data=config_path, # The path to your data.yaml file.
    epochs=50, # How many times the model will look at the entire dataset
    imgsz=640, # The resolution the images are resized to before entering the network
    batch=16, # How many images the model looks at before updating its internal weights
    overlap_mask=False, # When true, it merges masks into one layer. When false, each object has its own layer.
    mask_ratio=1,
    workers=2, # The number of CPU threads used to load and preprocess images
    device=0, # Use 'cpu' if you don't have an NVIDIA GPU
    name="handwriting_line_detector"
)

Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Line-Segmentation-6/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=handwriting_line_detector, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, ove

KeyboardInterrupt: 

### Test the YOLO Model

In [ ]:
# TEST 1
# Load your best trained model
model = YOLO("/content/drive/MyDrive/yolo_model/best.pt")

# Run prediction and save crops automatically
results = model.predict(source="/content/drive/MyDrive/OCR/output.png", save_crop=True, project="ocr_test", name="crops")



image 1/1 /content/drive/MyDrive/OCR/output.png: 640x640 13 Lines, 12.3ms
Speed: 9.0ms preprocess, 12.3ms inference, 45.1ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /content/runs/detect/ocr_test/crops


In [11]:
# TEST 2
# Better for working with TrOCR (without saving to directory)
def get_crops_for_ocr(image_path, model_path):
    model = YOLO(model_path)
    results = model.predict(source=image_path, conf=0.25)

    # Load the original image with OpenCV
    img = cv2.imread(image_path)
    crops = []

    # Access the detected boxes
    boxes = results[0].boxes.xyxy.cpu().numpy() # [x1, y1, x2, y2]

    # Sort boxes from top to bottom (y-axis) so the text stays in order
    boxes = boxes[boxes[:, 1].argsort()]

    for i, box in enumerate(boxes):
        x1, y1, x2, y2 = map(int, box)

        # Crop the image using slicing
        crop_img = img[y1:y2, x1:x2]

        # Convert BGR (OpenCV) to RGB (PIL/TrOCR)
        crop_pil = PILImage.fromarray(cv2.cvtColor(crop_img, cv2.COLOR_BGR2RGB))
        crops.append(crop_pil)

        # Optional: Save for debugging
        os.makedirs("/content/crop_outputs/",exist_ok=True)
        crop_pil.save(f"/content/crop_outputs/new_line_{i}.png")

    return crops

# Usage
line_crops = get_crops_for_ocr("/content/drive/MyDrive/OCR/test-output-1.png", "/content/drive/MyDrive/yolo_model/best.pt")
print(f"Detected {len(line_crops)} lines of text.")



image 1/1 /content/drive/MyDrive/OCR/test-output-1.png: 640x480 5 Lines, 46.7ms
Speed: 6.9ms preprocess, 46.7ms inference, 3.0ms postprocess per image at shape (1, 3, 640, 480)
Detected 5 lines of text.


### Testing the TROCR Handwritten model

In [ ]:
# Process the document image: fix orientation, perspective, lighting, and binarize
clean = process_document('/content/drive/MyDrive/OCR/pre-process test image.jpg')

# Save the cleaned image so you can visually inspect what the model "sees"
cv2.imwrite('/content/drive/MyDrive/OCR/test-output-1.png', clean)

#line segemtation


# Run OCR and print the result
text = run_ocr(clean)
print('Extracted text:', text)


### Full Test for the TROCR Handwritten model

In [14]:
# Process the document image: fix orientation, perspective, lighting, and binarize
clean = process_document('/content/drive/MyDrive/OCR/pre-process test image 3.jpg')

# Convert the 1-channel 'clean' image to a 3-channel BGR image for YOLO model compatibility
clean_bgr = cv2.cvtColor(clean, cv2.COLOR_GRAY2BGR)

# Save the cleaned 3-channel image to a temporary file
temp_image_path = '/content/temp_cleaned_image.png'
cv2.imwrite(temp_image_path, clean_bgr)

# Save the cleaned original (1-channel) image so you can visually inspect what the model "sees"
cv2.imwrite('/content/drive/MyDrive/OCR/test-output-6.png', clean)

#line segemtation
# Use the path to the 3-channel temporary image for line segmentation
line_crops = get_crops_for_ocr(temp_image_path, "/content/drive/MyDrive/yolo_model/best.pt")

full_text = ""
# Run OCR
for crop in line_crops:
  text = run_ocr(np.array(crop))
  full_text += (text + "\n")

print('Extracted text:', full_text)

# --- Integration with your existing loop ---
# Assuming 'full_text' is the string created by joining TrOCR lines[cite: 3]
cleaned_document = ollama_nlp_layer(full_text)

print("\n--- REFINED OUTPUT  ---")
print(cleaned_document)

# Clean up the temporary image file
import os
os.remove(temp_image_path)


No document contour found — skipping perspective correction

image 1/1 /content/temp_cleaned_image.png: 640x480 17 Lines, 10.0ms
Speed: 2.2ms preprocess, 10.0ms inference, 1.6ms postprocess per image at shape (1, 3, 640, 480)
Extracted text: Dear Teresa
Thankym for everything .
for the way you show up
every day , for how much
you carry ; for your calm .
strength , If for your beautiful
carins nature . You've
supported me more than
yin't ever knows ! I've
felt very lucky to work
alonside um't get to
know you better . I know
this is not the end , if
anything if the beginning
of a beautiful friendship .
" So much done .
Margui xx .


--- REFINED OUTPUT  ---
Dear Teresa  
Thank you for everything.  
for the way you show up  
every day, for how much  
you carry; for your calm  
strength, and for your beautiful  
caring nature. You've  
supported me more than I ever knew! I've  
felt very lucky to work  
alongside and get to know you better. I know  
this is not the end, if  
anything, it's 